# Investigation 3: Well Activity Radius Signal (Q3)

**Hypothesis:** Available blocks within 10–25 km of a recently spudded well receive bids at a higher rate in the subsequent sale than blocks in quiet areas. This is the empirical basis for the MVP 2 Treasure Toggle.

**Decision Rule:**
- Clear monotonic gradient, best combo p < 0.05 → Lock in best (radius, window) as canonical feature. Treasure Toggle proceeds.
- Gradient exists but weak → Keep Treasure Toggle, relax accuracy target (70% → 60%).
- No gradient → Remove Treasure Toggle from MVP 2. Replace with simple "Recent Activity" layer.

**Validation discipline:** Well locations and spud dates are observable pre-sale. December 2025 bid outcomes are held-out.

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from scipy.stats import chi2_contingency
from shapely.geometry import Point
import os

# Paths
SHAPEFILE_PATH = '../../data/shapefiles/blocks.shp'
SALE_DEC2025_DIR = '../data/sale_obbba_dec2025/'
WELLS_PATH = '../data/wells/boreholes.csv'

# Parameters to test
RADII_KM = [10, 25]
LOOKBACK_MONTHS = [6, 18]
WELL_COUNT_BINS = [0, 1, 2]  # 0, 1, 2+ wells

# Target CRS for distance computation
UTM_15N = 'EPSG:26915'

print('Configuration set.')

In [ ]:
# Load shapefile and reproject to UTM 15N
gdf_blocks = gpd.read_file(SHAPEFILE_PATH)
gdf_blocks_utm = gdf_blocks.to_crs(UTM_15N)
print(f'Loaded {len(gdf_blocks_utm)} blocks, reprojected to {UTM_15N}')

# Compute block centroids in UTM
gdf_blocks_utm['centroid'] = gdf_blocks_utm.geometry.centroid
print('Centroids computed.')

In [ ]:
# Load well data
# TODO: Uncomment when data is downloaded
# df_wells = pd.read_csv(WELLS_PATH, parse_dates=['spud_date'])
# print(f'Loaded {len(df_wells)} well records.')
#
# # Convert wells to GeoDataFrame in UTM 15N
# gdf_wells = gpd.GeoDataFrame(
#     df_wells,
#     geometry=gpd.points_from_xy(df_wells.longitude, df_wells.latitude),
#     crs='EPSG:4267'  # NAD27, same as blocks
# ).to_crs(UTM_15N)
# print(f'Wells reprojected to {UTM_15N}')

## 2. Compute Well Counts per Block

For each available block in Dec 2025, count wells spudded within {10 km, 25 km} of block centroid in {6-month, 18-month} lookback windows.

Result: 4 new columns per block — one for each (radius, window) combination.

In [ ]:
# TODO: Implement spatial well count
# Steps:
# 1. Define Dec 2025 sale bid deadline date
# 2. For each (radius, lookback_window) combination:
#    a. Filter wells by spud_date within the lookback window
#    b. For each available block centroid, count wells within radius
#       - Use gdf_wells.geometry.distance(block_centroid) < radius_m
#       - Or use spatial index (sindex) + buffer for efficiency
#    c. Store as new column: f'wells_{radius}km_{window}mo'
#
# Efficiency note: for ~65K blocks x ~N wells, consider using
# sjoin with buffered centroids rather than pairwise distance.
pass

## 3. Bin Blocks by Well Count

Create bins: `0 wells`, `1 well`, `2+ wells` for each (radius, window) combination.

In [ ]:
# TODO: Bin well counts
# def bin_well_count(n):
#     if n == 0: return '0 wells'
#     elif n == 1: return '1 well'
#     else: return '2+ wells'
#
# for radius in RADII_KM:
#     for window in LOOKBACK_MONTHS:
#         col = f'wells_{radius}km_{window}mo'
#         bin_col = f'well_bin_{radius}km_{window}mo'
#         df_blocks[bin_col] = df_blocks[col].apply(bin_well_count)
pass

---
## 4. VALIDATION — HELD-OUT (December 2025 Sale)

**⚠️ This section uses the held-out December 2025 bid outcomes for evaluation only.**
**Well locations and spud dates are pre-sale observable data.**

In [ ]:
# TODO: Compute bid rates per bin for each (radius, window) combination
#
# For each combination:
#   For each bin (0, 1, 2+):
#     bid_rate = blocks_that_received_bids / total_blocks_in_bin
#
# Test monotonic gradient: Cochran-Armitage trend test
# from scipy.stats import ??? # may need to implement manually
#
# Identify the combination with:
#   - Strongest lift (bid_rate_2plus / bid_rate_0)
#   - Lowest p-value for monotonic trend
pass

## 5. Lag Structure Test

The Treasure Toggle assumption: a well drilled *today* shifts bid behavior in the *next* sale.

Test: Does a well spudded 0–6 months before the sale have more predictive power than one spudded 7–18 months before? Or is the signal delayed?

In [ ]:
# TODO: Compare bid rate lift for:
# - Recent wells (0–6 months before sale)
# - Older wells (7–18 months before sale)
# Use the best radius from Section 4.
# Plot lag curve: lift vs. lookback window midpoint
pass

## 6. Treasure Toggle Preview Map

Pick a real historical well (from Sales 257/261 period). Show surrounding blocks.
Color by whether the block received a bid in the next sale.
Compare to what the radius model would predict.

In [ ]:
# TODO: Treasure Toggle preview
# 1. Pick a well spudded between Sales 257 and 261
# 2. Draw circles at 10km and 25km around it
# 3. Show which blocks within those circles received bids in Sale 261
# 4. Annotate: "If Treasure Toggle existed, it would have highlighted these blocks"
pass

## 7. Heatmaps

2x2 heatmap: rows = radius {10, 25 km}, columns = window {6, 18 months}.
Cell value = lift (bid_rate for 2+ wells bin / bid_rate for 0 wells bin).

In [ ]:
# TODO: Build heatmap
# lift_matrix = np.zeros((len(RADII_KM), len(LOOKBACK_MONTHS)))
# for i, radius in enumerate(RADII_KM):
#     for j, window in enumerate(LOOKBACK_MONTHS):
#         lift_matrix[i, j] = bid_rate_2plus[radius][window] / bid_rate_0[radius][window]
#
# fig, ax = plt.subplots(figsize=(8, 6))
# im = ax.imshow(lift_matrix, cmap='YlOrRd', aspect='auto')
# ax.set_xticks(range(len(LOOKBACK_MONTHS)))
# ax.set_xticklabels([f'{w} months' for w in LOOKBACK_MONTHS])
# ax.set_yticks(range(len(RADII_KM)))
# ax.set_yticklabels([f'{r} km' for r in RADII_KM])
# ... annotate cells with lift values ...
# plt.colorbar(im, label='Lift (2+ wells / 0 wells)')
# ax.set_title('Q3: Well Activity Lift by (Radius, Lookback Window)')
# plt.savefig('../outputs/q3_well_activity_heatmap.png', dpi=150, bbox_inches='tight')
pass

## 8. Findings

### Results
- **Best (radius, window) combination:** [TBD]
- **Lift at best combination:** [TBD]x
- **Monotonic gradient significant:** [TBD]
- **Lag structure:** [immediate / delayed / unclear]

### Interpretation
[TBD — fill in based on results and PRD decision rule]

### Recommendation
[TBD — Lock in canonical feature / Relax accuracy target / Remove Treasure Toggle]